In [ ]:
import pandas as pd
import numpy as np
import re
import string
from collections import Counter
from datetime import datetime
import nltk
import textstat
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

tqdm.pandas()


nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('vader_lexicon', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk import pos_tag
STOPWORDS = set(stopwords.words('english'))
import torch
from sentence_transformers import SentenceTransformer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")
from google.colab import drive

# 1. Mount Drive (Essential for accessing your saved data.zip)
drive.mount('/content/drive')

In [ ]:
import os
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds/featuring'
CLEANED_DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'


In [ ]:


# 1. Prepare masks and data
train_mask = (df['split'] == 'train')
valtest_mask = ~train_mask

# Use the 'safe_content' we created during the OpenAI step
train_texts = df.loc[train_mask, 'safe_content'].tolist()
valtest_texts = df.loc[valtest_mask, 'safe_content'].tolist()

# Ensure we are using the new OpenAI embeddings matrix
train_embeddings = embeddings[train_mask.values]
valtest_embeddings = embeddings[valtest_mask.values]

# 2. Configure Sub-models
# We keep these settings, but UMAP will now handle the 1536-dim OpenAI input
umap_model = UMAP(n_neighbors=15, n_components=10, min_dist=0.0, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, min_samples=10, prediction_data=True)

# 3. Initialize BERTopic
# We set embedding_model=None because we are providing the embeddings manually
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    embedding_model=None, 
    verbose=True,
    nr_topics='auto',
)

# 4. Run Topic Modeling
# Pass the pre-computed OpenAI embeddings directly
print("Fitting BERTopic on training embeddings...")
train_topics, train_probs = topic_model.fit_transform(train_texts, embeddings=train_embeddings)

print("Transforming validation/test data...")
valtest_topics, valtest_probs = topic_model.transform(valtest_texts, embeddings=valtest_embeddings)

# 5. Assign results back to DataFrame
df.loc[train_mask, 'topic_id'] = train_topics
df.loc[valtest_mask, 'topic_id'] = valtest_topics
df['topic_id'] = df['topic_id'].astype(int)

# 6. Reporting
print(f"\nNo. of topics found: {len(set(train_topics)) - (1 if -1 in train_topics else 0)}")
print(f"Outlier texts in train: {(df.loc[train_mask, 'topic_id'] == -1).sum()}")
print(f"Outlier texts in val+test: {(df.loc[valtest_mask, 'topic_id'] == -1).sum()}")

topic_info = topic_model.get_topic_info().head(11)
print(topic_info[['Topic', 'Count', 'Name']].to_string())

# 7. Save the model
# Since we aren't using a local ST model, we don't save st_model here
topic_model.save(
    f"{DATA_DIR}/bertopic_openai_model", 
    serialization="safetensors", 
    save_ctfidf=True
)

In [ ]:

# Updated meta_cols to reflect new schema
meta_cols = ['text', 'text_clean', 'author', 'created_utc', 'timestamp', 'subreddit',
             'source', 'label', 'split', 'interaction_type', 'id', 'post_id']
emb_cols_list = [c for c in df.columns if c.startswith('emb_')]
feature_cols = [c for c in df.columns if c not in meta_cols and c not in emb_cols_list]

print(f"Total tabular features: {len(feature_cols)}")
print(f"Embedding features: {len(emb_cols_list)}")
print(f"Total features (tabular + embeddings): {len(feature_cols) + len(emb_cols_list)}")
print(f"\nTabular feature columns:\n{feature_cols}")
print(f"\nDataset shape: {df.shape}")


In [ ]:

comparison = df.groupby('label')[feature_cols].mean().T
comparison.columns = ['Human (0)', 'AI (1)']
comparison['AI/Human Ratio'] = comparison['AI (1)'] / (comparison['Human (0)'] + 1e-10)
comparison = comparison.sort_values('AI/Human Ratio', ascending=False)

print("Top features where AI > Human:")
print(comparison.head(15).to_string())
print("\nTop features where Human > AI:")
print(comparison.tail(15).to_string())

In [ ]:
df['created_utc'] = df['created_utc'].astype(str)

tabular_cols = feature_cols + ['label', 'source', 'split']
all_feature_cols = feature_cols + emb_cols_list + ['label', 'source', 'split']

for split_name in ['train', 'val', 'test']:
    split_df = df[df['split'] == split_name].copy()
    split_df = split_df.reset_index(drop=True)
    
    # 1. Full Dataset with text metadata
    out_full = f"{DATA_DIR}/features_subset_{split_name}.parquet"
    split_df.to_parquet(out_full, index=False)
    
    # 2. Tabular Features Only
    out_tab = f"{DATA_DIR}/feature_matrix_subset_{split_name}.parquet"
    split_df[tabular_cols].to_parquet(out_tab, index=False)
    
    # 3. Tabular + Embeddings
    out_all = f"{DATA_DIR}/feature_matrix_full_subset_{split_name}.parquet"
    split_df[all_feature_cols].to_parquet(out_all, index=False)
    
    print(f"[{split_name.upper()}] Saved {len(split_df)} rows. Label dist: {split_df['label'].value_counts().to_dict()}")
